# **Kaggle Setup**

In [ ]:
!pip install -q kaggle

In [ ]:
from google.colab import files
files.upload()

In [ ]:
! mkdir ~/.kaggle

In [ ]:
! cp kaggle.json ~/.kaggle/

In [ ]:
! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# ! kaggle datasets list
# ! unzip ml-spark.zipnn

In [ ]:
! kaggle competitions download -c ml-spark

# **Section : 01**

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.subplots as sp
import plotly.graph_objects as go
from plotly.subplots import make_subplots


import warnings
warnings.filterwarnings("ignore")

In [ ]:
df_raw = pd.read_csv("train.csv")
df = df_raw.drop(["id"],axis=1)

In [ ]:
def info_data(data):
    feature_name = data.columns
    isnull, number_unique_value = list(),list()

    feature_dtype = data.dtypes
    for i in data.columns:
        isnull.append(data[i].isnull().sum())
        number_unique_value.append(len(data[i].unique()))
    info = pd.DataFrame([feature_name,feature_dtype,number_unique_value,isnull]).T
    info.columns = ["Features Name",'Feature Data Type','Number Unique Value','Number of Null Value']
    print(f"Dimensition of DataSet is :--> \n\tNumber of Observation/Datapoints {data.shape[0]} \n\tNumbers Of Features : {data.shape[1]}\n")
    print(f"Total Duplicate Data Point is : {data.duplicated().sum()}\n")
    return info

info_data(df)

## **Sub-Section 1.1 : Analysis**

### **Univariate Analysis**

#### Categorical Data

In [ ]:
fig = sp.make_subplots(rows = 2, cols = 3, subplot_titles =("Job Distribution","Education","Month Wise Datapoints ",  "Marital Distribution","Loan Approval Status","Housing Distribution"))


# Plot 1 :: Job Distribution
job_counts = df["job"].value_counts().reset_index()
job_counts.columns = ["Job Categories", "Count"]
fig.add_trace(
    go.Bar(x=job_counts["Job Categories"],
           y=job_counts["Count"],
           marker_color=job_counts["Count"],
           showlegend=True,), row=1, col=1)


# Plot 2 :: Distribution education
education_counts = df["education"].value_counts().reset_index()
education_counts.columns = ["Education Categories", "Count"]
fig.add_trace(
    go.Bar(x=education_counts["Education Categories"],
           y=education_counts["Count"],
           marker_color=education_counts["Count"],
           showlegend=True,), row=1, col=2)


# Plot 3 :: Month Wise Datapoints
month_counts = df["month"].value_counts().reset_index()
month_counts.columns = ["Month Categories", "Count"]
colors = [
    "blue", "green", "orange", "red", "purple",
    "cyan", "lime", "yellow", "pink", "teal",
    "magenta", "gray"
]

fig.add_trace(
    go.Bar(
        x=month_counts["Month Categories"],
        y=month_counts["Count"],
        marker=dict(color=colors),
        showlegend=True
    ),
    row=1, col=3
)


# Plot 4::  Marital Distribution
#  Data for marital column
marital_counts = df['marital'].value_counts()
fig.add_trace(
    go.Bar(x=marital_counts.index, y=marital_counts.values, marker=dict(color=["blue", "red", "green", "purple", "orange"])),
    row=2, col=1)

# Plot 5:: Loan Approval Status
loan_counts = df["loan"].value_counts().reset_index()
loan_counts.columns = ["loan Categories", "Count"]
fig.add_trace(
    go.Bar(x=loan_counts["loan Categories"],
           y=loan_counts["Count"],
           marker=dict(color=["blue", "red"]),
          #  marker_color=job_counts["Count"],
           showlegend=True,), row=2, col=2)

# plot 6:: Housing Data Points
housing_counts = df['housing'].value_counts()
fig.add_trace(
    go.Bar(x=housing_counts.index, y=housing_counts.values, marker=dict(color=["pink", "yellow", "cyan", "lime", "gray"])),
    row=2, col=3)

fig.update_layout(title_text = "Descriptive Analysis", height = 800, width = 1400, showlegend = False)

#### Numerical Features

In [ ]:
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=["Balance", "Day", "Duration", "Campaign"])

# Add histograms for each variable
fig.add_trace(go.Histogram(x=df['balance'], name='Balance'), row=1, col=1)
fig.add_trace(go.Histogram(x=df['day'], name='Day'), row=1, col=2)
fig.add_trace(go.Histogram(x=df['duration'], name='Duration'), row=2, col=1)
fig.add_trace(go.Histogram(x=df['campaign'], name='Campaign'), row=2, col=2)

fig.update_layout(title="Distributions of Given Variables", height=800, showlegend=False)
fig.show()

### **Bivariate Analysis**

**Relation Of Campaign and Target**

In [ ]:
campaing_wise_target = dict()

for i in df["campaign"].unique():
  counts = df[df["campaign"] == i].groupby(["Target"])["Target"].count()
  campaing_wise_target[i] = {
      "Class 0": counts.get(0, 0),  # Use get() with default value 0
      "Class 1": counts.get(1, 0)   # Use get() with default value 0
  }
# pd.DataFrame(campaing_wise_target)
# print(campaing_wise_target)

# Extract X-axis (keys) and Y-axis (values)
x_values = list(campaing_wise_target.keys())
class_0_values = [campaing_wise_target[k]["Class 0"] for k in x_values]
class_1_values = [campaing_wise_target[k]["Class 1"] for k in x_values]

# Create Plotly Figure
fig = go.Figure()

# Add lines for Class 0 and Class 1
fig.add_trace(go.Scatter(x=x_values, y=class_0_values, mode="lines+markers", name="Class 0"))
fig.add_trace(go.Scatter(x=x_values, y=class_1_values, mode="lines+markers", name="Class 1"))

# Update layout
fig.update_layout(
    title="Comparison of Classes",
    xaxis_title="Keys",
    yaxis_title="Values",
    legend_title="Classes",
)

# Show the figure
fig.show()


### **Multivariate Analysis**

In [ ]:
temp_list = list()
job_wise_data_raw = {"job_title":["Primary","Secondary","Tertiary","Unknown","Divorced",
                                          "Married","Single","Having Housing Loan","Not Having Housing Loan","Having Personal Loan"
                                          ,"Not Having Personal Loan","No of Defaulter","No of Non_Defaulter","Not subscribed","subscribed"]}

for i in df.groupby(["job"])["Target"].count().keys():
    job_df = df[df["job"]==i]

    job_wise_data_raw[i] = list(job_df.groupby(["education"])["education"].count())+list(job_df.groupby(["marital"])["marital"].count()) + list(job_df.groupby(["housing"])["housing"].count()) + list(job_df.groupby(["loan"])["loan"].count()) +list(job_df.groupby(["default"])["default"].count())+list(job_df.groupby(["Target"])["Target"].count())


job_wise_data = pd.DataFrame(job_wise_data_raw).T
job_wise_data.columns = job_wise_data.iloc[0]
job_wise_data = job_wise_data[1:]
job_wise_data

plt.figure(figsize=(12, 10))
#  cmap = "BuPu" ,"coolwarm", "viridis",
sns.heatmap(job_wise_data.corr(),annot=True, fmt=".2f", cmap="BuPu", linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title("Correlation Heatmap", fontsize=18)
plt.xticks(fontsize=12, rotation=45)
plt.yticks(fontsize=12, rotation=0)
plt.show()

#### Strong Relationships Between Loan Types and Marital Status:
     Married individuals have a strong positive correlation with having a housing loan (0.91) and a personal loan (0.93). This implies they are more likely to avail such loans.

     Conversely, single individuals correlate positively with not having housing loans (0.85) or personal loans (0.83).

#### Interdependence Between Housing and Personal Loans:
    There is a strong positive correlation (0.91) between having a housing loan and a personal loan. This suggests that individuals taking one type of loan are more likely to take the other.

#### Education Level and Loan Behavior:
    Individuals with tertiary education show a positive correlation with being married (0.87) and having housing loans (0.64). This indicates financial behaviors tied to higher educational qualifications.
    Other education levels, such as primary or secondary, show varying weaker correlations with loan statuses.

#### Non-Defaulters and Loan Status:
    Non-defaulters have a strong positive correlation with being married (0.98) and holding both housing and personal loans. This could imply better financial management by married individuals or dual-income households.

#### Subscription Status Links:
    Subscribing to services or programs has a positive correlation with having loans—housing loans (0.84) and personal loans (0.91)—suggesting that loan holders are more engaged with financial services or products.

In [ ]:
plt.figure(figsize=(12, 10))
#  cmap = "BuPu" ,"coolwarm", "viridis",
numerical_df = df.select_dtypes(include=np.number)
sns.heatmap(numerical_df.corr(),annot=True, fmt=".2f", cmap="viridis", linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title("Correlation Heatmap", fontsize=18)
plt.xticks(fontsize=12, rotation=45)
plt.yticks(fontsize=12, rotation=0)
plt.show()

## **Sub - Section 1.2 : EDA Given Questions**

##### **Question 01 :Customer Demographics & Behavior** <br> <hr>
- How do age, job type, marital status, and education level influence response rates? <br>
- Are certain customer segments more likely to respond than others?

**1. How do age, job type, marital status, and education level influence response rates?**

In [ ]:
# Define lists for unique values
marital_statuses = ['married', 'single', 'divorced']
education_levels = ['secondary', 'primary', 'tertiary', 'unknown']

# Define age ranges
age_ranges = [(18, 28), (28, 38), (38, 48), (48, 58), (58, 68), (68, 78), (78, 88)]

# Create a list to store the summarized data for plotting
result_raw = []

# Iterate over age ranges
for age_range in age_ranges:
    age_min, age_max = age_range
    age_key = f"{age_min}-{age_max}"  # Create a key for the age range

    # Filter dataset by age range
    age_filtered_data = df[(df["age"] >= age_min) & (df["age"] < age_max)]

    # Iterate through marital status and education levels
    for marital in marital_statuses:
        for education in education_levels:
            # Filter dataset by marital status and education level
            duration_mean = age_filtered_data[
                (age_filtered_data["marital"] == marital) &
                (age_filtered_data["education"] == education)
            ]['duration'].mean()

            # Append to plot_data
            result_raw.append({
                'Age Range': age_key,
                'Marital Status': marital,
                'Education Level': education,
                'Count': np.round(duration_mean, 2)
            })

# Convert the data into a DataFrame
result = pd.DataFrame(result_raw)
result.T

#### **Question 2 Communication Channels & Timing**<br><hr>
- Which contact method (cellular, telephone, unknown) has the highest response rate?
- Are certain months or weekdays more effective for outreach?

In [ ]:
temp_data = {"Features":["Cellular","Telephone","Unknown"],
             "Avarage Duration":list(df.groupby(["contact"])['duration'].mean()),
             "Maximum Duration":list(df.groupby(["contact"])['duration'].max()),
             "Minimum Duration": list(df.groupby(["contact"])['duration'].min())}
pd.DataFrame(temp_data)

# Observation : Telephones are more reliable than cellular devices for connecting with customers.

In [ ]:
month_wise_avg_duration = dict()
for i in df['month'].unique():
  month_wise_avg_duration[i] = {
      "Average_Duration":
round(float(df[df["month"]==i].groupby(["month"])["duration"].mean()),2),"STD_Duration":
round(float(df[df["month"]==i].groupby(["month"])["duration"].std()),2),"Maximum_Duration":
round(float(df[df["month"]==i].groupby(["month"])["duration"].max()),2),"Minimum_Duration":
round(float(df[df["month"]==i].groupby(["month"])["duration"].min()),2)
   }
pd.DataFrame(month_wise_avg_duration).T

# print(month_wise_avg_duration)

# Extract data for plotting
months = list(month_wise_avg_duration.keys())
average = [month_wise_avg_duration[month]['Average_Duration'] for month in months]
std_dev = [month_wise_avg_duration[month]['STD_Duration'] for month in months]
maximum = [month_wise_avg_duration[month]['Maximum_Duration'] for month in months]
minimum = [month_wise_avg_duration[month]['Minimum_Duration'] for month in months]

# Create the figure
fig = go.Figure()

# Add traces for each metric
fig.add_trace(go.Scatter(x=months, y=average, mode='lines+markers', name='Average', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=months, y=std_dev, mode='lines+markers', name='STD', line=dict(color='orange')))
fig.add_trace(go.Scatter(x=months, y=maximum, mode='lines+markers', name='Maximum', line=dict(color='green')))
fig.add_trace(go.Scatter(x=months, y=minimum, mode='lines+markers', name='Minimum', line=dict(color='red')))

# Update layout
fig.update_layout(
    title='Comparison of Durations by Month',
    xaxis_title='Months',
    yaxis_title='Duration',
    legend_title='Metrics',
    template='plotly'
)
# Show the plot
fig.show()


#### Observations
1. Duration peaked in December with an average of **340.80** and a maximum of **2062.0**.
2. Minimum durations were consistently low across months, ranging from **0.0** to **30.0**.
3. November had the highest maximum duration at **4918.0**, showing significant variation from its average.
4. Standard deviations suggest high variability in duration, especially in October (**301.78**) and December (**304.95**).
5. Overall, April and December show contrasting extremes—April with high averages and December with high peaks.

# **Section : 02**

In [ ]:
# load data
df_raw = pd.read_csv("/content/train.csv")
df = df_raw.drop(["id"],axis=1)

#### **Section : 2.1: To Check Outlier**
**balance and duration have a right-skewed distribution; outliers are replaced with the median value.**

In [ ]:
# Select numerical columns (int and float types)
l = [i for i in df.columns if df[i].dtype in ['int64', 'float64']]
numerical_data = df[l]

# Melt the DataFrame for visualization
df_melted = numerical_data.melt(var_name='Feature', value_name='Value')

# Create a combined boxplot using Plotly
import plotly.express as px
fig = px.box(df_melted, x='Feature', y='Value', title="Combined Boxplot for All Features")
fig.show()

In [ ]:
def detect_outliers(series):
    # Calculate Q1 (25th percentile) and Q3 (75th percentile)
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)

    # Calculate the Interquartile Range (IQR)
    IQR = Q3 - Q1

    # Define outlier boundaries
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Filter out the outliers
    outliers = series[(series < lower_bound) | (series > upper_bound)]
    return outliers

balance_outlier = list(set(detect_outliers(df['balance'])))
duration_outlier = list(set(detect_outliers(df['duration'])))

# Replace outliers in "balance" column
df["balance"] = df["balance"].apply(lambda x: df["balance"].median() if x in balance_outlier else x)

# Replace outliers in "duration" column
df["duration"] = df["duration"].apply(lambda x: df["duration"].median() if x in duration_outlier else x)

#### **Section : 2.2: Label Encoding**

In [ ]:
df["job"] = df["job"].map({'admin.': 1, 'technician': 2, 'housemaid': 3, 'services': 4,'entrepreneur': 5, 'blue-collar': 6, 'self-employed': 7, 'management': 8, 'retired': 9, 'unemployed': 10, 'student': 11, 'unknown': 12})

df['marital'] = df['marital'].map({'married':1, 'single':0, 'divorced':1})

df["education"] = df["education"].map({'primary':0, 'secondary':1, 'tertiary':2, 'unknown':3})

df["default"] = df["default"].map({'no':0, 'yes':1})
df["housing"] = df["housing"].map({'no':0, 'yes':1})
df["loan"] = df["loan"].map({'no':0, 'yes':1})
df['contact'] = df['contact'].map({'unknown':2, 'cellular':1, 'telephone':3})

df["poutcome"] = df["poutcome"].map({'unknown': 0, 'other': 1, 'failure': 2, 'success': 3})

df['month'] = df['month'].map({'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5,'jun': 6,'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10,'nov': 11, 'dec': 12})


In [ ]:
print(f" Traing Data Feature {[i for i in df.columns if i in test_data.columns and test_data[i].dtypes == 'object']}")

#### **Section : 2.3: Scaling**

In [ ]:
# Create Dependent and Indepdendent Features
X = df.drop("Target",axis = 1)
y = df["Target"]

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Apply the scaler to the DataFrame
scaled_data = scaler.fit_transform(X)

# Create a new DataFrame with the scaled data
scaled_data = pd.DataFrame(scaled_data, columns=X.columns)

# scaled_data

#### **Section : 2.4: Train - Test Split**

In [ ]:
from sklearn.model_selection import train_test_split

# Splitting the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X_train:\t", X_train.shape)
print("y_train:\t", y_train.shape)
print("X_tes  :\t", X_test.shape)
print("y_test :\t", y_test.shape)

#### **Section : 2.5: Train Multiple Model**

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier


# List of models to evaluate
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Support Vector Machine": SVC(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "AdaBoost": AdaBoostClassifier()
}

# Dictionary to store precision, recall, and F1 scores
results = []

# Train and evaluate each model
for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    results.append({"Model": model_name, "Precision": precision, "Recall": recall, "F1-Score": f1})

# Convert results to a DataFrame for better readability
results_df = pd.DataFrame(results)

# Sort by F1-score, precision, and recall for comparison
results_df = results_df.sort_values(by=["F1-Score", "Precision", "Recall"], ascending=False)
results_df